## Bronze Layer — Raw Data Ingestion
Auto Loader batch ingestion of the **arXiv metadata snapshot** (~5.22 GB, JSON Lines format) into a Delta table.

**Source:** `/Volumes/arxivist_ai_dev/raw/external_data/data_file/`  
**Target:** `arxivist_ai_dev.bronze.arxiv_metadata`  
**Method:** Auto Loader (`cloudFiles`) with `Trigger.AvailableNow` (batch mode)

In [0]:
# ---- Configuration ----
CATALOG = "arxivist_ai_dev"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "/Volumes/arxivist_ai_dev/raw/external_data/data_file/"
CHECKPOINT_PATH = "/Volumes/arxivist_ai_dev/bronze/checkpoints/arxiv_metadata"
SCHEMA_PATH = "/Volumes/arxivist_ai_dev/bronze/checkpoints/arxiv_metadata_schema"
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.arxiv_metadata"

In [0]:
# Create a managed volume for checkpoints if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.checkpoints")

# Ensure checkpoint directories exist
dbutils.fs.mkdirs(CHECKPOINT_PATH)
dbutils.fs.mkdirs(SCHEMA_PATH)

print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Schema path:     {SCHEMA_PATH}")

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Read with Auto Loader (cloudFiles) — batch mode
df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    .load(SOURCE_PATH)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

# Write as Delta table — Trigger.AvailableNow processes all files then stops
query = (
    df_bronze.writeStream
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()
print(f"Bronze ingestion complete → {BRONZE_TABLE}")

In [0]:
# Verify the bronze table
df_check = spark.table(BRONZE_TABLE)

print(f"Total rows:    {df_check.count():,}")
print(f"Total columns: {df_check.columns}")
print("\nSchema:")
df_check.printSchema()
print("\nSample data:")
display(df_check.limit(5))